In [2]:
import hdbscan  # 使用独立的 hdbscan 库
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from Bio.Seq import Seq
import numpy as np 
import pandas as pd

def kmer_embedding(sequences, k=4, min_cluster_size=10, min_samples=None):
    """
    将序列转换为 k-mer 频率向量并使用 HDBSCAN 聚类
    """
    def get_kmers(seq, k):
        kmers = []
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            rc_kmer = str(Seq(kmer).reverse_complement())
            canonical = min(kmer, rc_kmer)
            kmers.append(canonical)
        return ' '.join(kmers)
    
    print("1. 正在提取 k-mer 特征...")
    kmer_docs = [get_kmers(seq, k) for seq in sequences]
    
    # 1. 向量化
    vectorizer = CountVectorizer()
    X_sparse = vectorizer.fit_transform(kmer_docs)
    
    print(f"2. 数据矩阵形状: {X_sparse.shape}")
    
    # 2. 转换为密集矩阵并归一化
    # 解释：k=4 时特征只有 4^4=256 维，50万行 x 256列 的密集矩阵仅占用约 1GB 内存
    # 这是完全安全的，且对于后续使用 KDTree 至关重要
    print("3. 转换为密集矩阵并归一化 (L2)...")
    X_dense = X_sparse.toarray()
    X_normalized = normalize(X_dense, norm='l2')
    
    # 3. 聚类
    print("4. 开始 HDBSCAN 聚类...")
    # 使用 metric='euclidean'，因为数据已经 L2 归一化
    # algorithm='kdtree' 会构建树结构，避免计算距离矩阵，速度极快且省内存
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='euclidean',  # 关键：使用欧氏距离
        algorithm='best',  # 关键：使用 KDTree 加速，避免 O(N^2) 内存
#         n_jobs=-1            # 使用所有 CPU 核心
#         core_dist_n_jobs=-1
    )
    
    clusterer.fit(X_normalized)
    
    print("5. 聚类完成！")
    
    # 输出结果
    results = {}
    for seq, label in zip(sequences, clusterer.labels_):
        results[seq] = label
    
    return results, vectorizer.get_feature_names_out()

# --- 运行代码 ---
df = pd.read_csv('R4/pairwise_motifs_parasail_revcomp.tsv', sep='\t')
sequences = df[(df['Match_Length']>=6) & (df['Match_Length']<=20)]['Query_Seq'].tolist()

# 运行
labels, features = kmer_embedding(sequences, k=4)

# 统计结果
n_clusters = len(set(labels.values())) - (1 if -1 in labels.values() else 0)
n_noise = list(labels.values()).count(-1)

print(f"\n=== 结果统计 ===")
print(f"K-mer 特征数量: {len(features)}")
print(f"发现的簇数量: {n_clusters}")
print(f"噪声点数量: {n_noise} ({n_noise/len(labels)*100:.2f}%)")

# 将字典转换为 DataFrame
# orient='index' 表示字典的键作为行索引，值作为列数据
df = pd.DataFrame.from_dict(labels, orient='index', columns=['cluster_label'])

# 重置索引，把序列从“索引”变成普通的“第一列”
df.reset_index(inplace=True)
df.rename(columns={'index': 'sequence'}, inplace=True)

# 保存为 CSV (推荐，Excel 可打开)
# df.to_csv('cluster_results_pandas.csv', index=False)

# 或者保存为 TSV (生物信息学常用)
df.to_csv('cluster_results_pandas.tsv', sep='\t', index=False)

print("保存成功！前 5 行预览：")
print(df.head())

1. 正在提取 k-mer 特征...
2. 数据矩阵形状: (496717, 136)
3. 转换为密集矩阵并归一化 (L2)...
4. 开始 HDBSCAN 聚类...
5. 聚类完成！

=== 结果统计 ===
K-mer 特征数量: 136
发现的簇数量: 6886
噪声点数量: 242586 (89.37%)


In [20]:
len(labels)

271449

In [28]:
df

,sequence,cluster_label
0,TTTTGA,5474
1,CTATCC,303
2,CATAGGGAAA,-1
3,AAGCTCAGAGTGATT,-1
4,TATCTT,647
...,...,...
271444,AAAAAAAAAAGCAGC,-1
271445,TTCCAAAATATAA,-1
271446,GTTTCCCCTAATG,-1
271447,GAAACTAAGT,-1
